# Comparing Models: the 'shortlist and test' process

The key skill is NOT memorizing which model to use. It's: **train a few candidates on the SAME data and let cross-validation pick the winner.** This notebook shows exactly that, so you can stop worrying about 'knowing' the right model in advance.

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Step 1: Get a real dataset (breast cancer classification)

A built-in dataset: predict whether a tumor is malignant or benign from 30 features. Real, messy, multi-feature data — perfect for comparing models.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target
print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
print(f'Classes: {data.target_names}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 2: Shortlist several candidate models

Notice every model uses the EXACT same interface. We just put them in a dictionary and loop.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

candidates = {
    'Logistic Regression': LogisticRegression(max_iter=5000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42),
}

## Step 3: Let cross-validation score each one

The DATA decides the winner — not your memory. This is the whole point.

In [ ]:
from sklearn.model_selection import cross_val_score

print(f"{'Model':<22} {'CV accuracy':>12}")
print('-' * 36)
results = {}
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = scores.mean()
    print(f'{name:<22} {scores.mean():>11.3f}')

winner = max(results, key=results.get)
print(f'\nWinner by cross-validation: {winner}')

## Step 4: Take the winner, evaluate ONCE on the untouched test set

Final, honest performance on data no model has seen.

In [ ]:
from sklearn.metrics import classification_report

best_model = candidates[winner]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f'Final test report for {winner}:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

## The takeaway

- You did NOT need to 'know' the best model in advance.
- You shortlisted a few, trained them with identical code, and **cross-validation picked the winner**.
- This exact loop works for almost any problem — just swap the dataset and the candidate models.

## Your turn

1. Add `Ridge`-style regularization strength: try `LogisticRegression(C=0.1)` vs `C=10`. Does the CV score change?
2. Swap in the `load_wine` or `load_iris` dataset. Does the winning model change? (It often does — proving there's no single 'best' model.)
3. Add `scoring='f1_macro'` instead of accuracy. Does the ranking shift?